# Photos Library Duplicate Cleanup Notebook

Report-only v1. This notebook does **not** delete anything.

Run order:
1. Run configuration.
2. Load/build inventory.
3. Fill identity fields.
4. Group and analyze duplicate candidates.
5. Write permanent operation report.

Helper functions live in `photos_duplicate_cleanup_helpers.py` so the notebook stays readable.


In [1]:
# ============================================================
# Cell 1. Configuration
# ============================================================

from pathlib import Path
import os
import sys
import json
from datetime import datetime

PROJECT_ROOT = Path("/Users/huohsien/workspace/python/explore_photos_library")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Photos Library registry
# ------------------------------------------------------------

PHOTOS_LIBRARY_PATHS = {
    "current_default": (
        "/Users/huohsien/Pictures/"
        "Photos Library.photoslibrary"
    ),
    "backup_20250317": (
        "/Volumes/NEW-PRO-G40--20250315/"
        "Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary"
    ),
    "test": (
        "/Users/huohsien/Pictures/"
        "test.photoslibrary"
    ),
}

# ------------------------------------------------------------
# Choose target library here.
#
# Valid examples:
#   "current_default"
#   "backup_20250317"
#   "test"
# ------------------------------------------------------------

TARGET_LIBRARY_ID = "backup_20250317"

if TARGET_LIBRARY_ID not in PHOTOS_LIBRARY_PATHS:
    raise KeyError(
        f"Unknown TARGET_LIBRARY_ID: {TARGET_LIBRARY_ID!r}\n"
        f"Available library ids: {sorted(PHOTOS_LIBRARY_PATHS)}"
    )

LIBRARY_ID = TARGET_LIBRARY_ID
LIBRARY_PATH = Path(PHOTOS_LIBRARY_PATHS[TARGET_LIBRARY_ID])

CACHE_DIR = PROJECT_ROOT / "data" / "inventory_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NAME = LIBRARY_ID
INVENTORY_CACHE_PATH = CACHE_DIR / f"{CACHE_NAME}.inventory.pkl.gz"

REPORTS_ROOT = PROJECT_ROOT / "IMPORTANT_Photos_Library_Critical_Operation_Reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
REPORT_DIR = REPORTS_ROOT / f"{RUN_TIMESTAMP}__Photos_Library_Duplicate_Cleanup__{LIBRARY_ID}"

print("TARGET_LIBRARY_ID:", TARGET_LIBRARY_ID)
print("LIBRARY_ID:", LIBRARY_ID)
print("LIBRARY_PATH:", LIBRARY_PATH)
print("LIBRARY_PATH exists:", LIBRARY_PATH.exists())
print("INVENTORY_CACHE_PATH:", INVENTORY_CACHE_PATH)
print("REPORT_DIR will be created only when writing report:", REPORT_DIR)

if not LIBRARY_PATH.exists():
    raise FileNotFoundError(f"Photos Library path does not exist: {LIBRARY_PATH}")

TARGET_LIBRARY_ID: backup_20250317
LIBRARY_ID: backup_20250317
LIBRARY_PATH: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
LIBRARY_PATH exists: True
INVENTORY_CACHE_PATH: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
REPORT_DIR will be created only when writing report: /Users/huohsien/workspace/python/explore_photos_library/IMPORTANT_Photos_Library_Critical_Operation_Reports/20260608-122355__Photos_Library_Duplicate_Cleanup__backup_20250317


In [2]:
# ============================================================
# Cell 2. Load or build inventory
# ============================================================

import osxphotos
from photos_inventory import (
    build_inventory,
    print_inventory_summary,
    save_inventory_cache,
    load_inventory_cache,
)

FORCE_REBUILD_INVENTORY = True

if INVENTORY_CACHE_PATH.exists() and not FORCE_REBUILD_INVENTORY:
    inventory = load_inventory_cache(
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )
    print("Loaded inventory cache:", INVENTORY_CACHE_PATH)
else:
    print("Building inventory from Photos Library:")
    print(LIBRARY_PATH)

    photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))
    osx_assets = photosdb.photos()

    inventory = build_inventory(osx_assets)

    save_inventory_cache(
        inventory=inventory,
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )

    print("Saved inventory cache:", INVENTORY_CACHE_PATH)

print_inventory_summary(inventory)


Building inventory from Photos Library:
/Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
processed assets: 10000
processed assets: 20000
processed assets: 30000
processed assets: 40000
processed assets: 50000
processed assets: 60000
processed assets: 70000
saved inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
elapsed seconds: 0.71
Saved inventory cache: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz
inventory assets: 71599
inventory albums: 5172
inventory folders: 35
movies: 6240
hidden: 0
favorites: 699
descriptions: 727
keywords: 23758


In [3]:
# ============================================================
# Cell 3. Fill identity fields
# ============================================================

from photos_duplicate_cleanup_helpers import fill_duplicate_cleanup_identity_fields

fill_duplicate_cleanup_identity_fields(inventory)


Filled identity fields
asset count: 71599
base_id filled: 71599
unique_id filled: 71599

First 3 assets after fill:
--------------------------------------------------------------------------------
original_filename: IMG_0239.PNG
date: 2020-12-01T13:32:44+08:00
path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/A/ABB24AF2-C25A-4301-8B0F-6A17215C8355.png
file_size_bytes: 5964163
base_id: ('IMG_0239.PNG', '12-01 13:32:44.000000', 5964163)
unique_id: (('IMG_0239.PNG', '12-01 13:32:44.000000', 5964163), (('description', None), ('keywords', ('HIDE', 'NSFW', 'NSFW_IG')), ('favorite', False), ('hidden', False), ('album_titles', ('IG Pretty Girls',)), ('folder_paths', ('NSFW',))))
--------------------------------------------------------------------------------
original_filename: IMG_2676.JPG
date: 2023-02-09T16:02:13.679000+08:00
path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250

In [4]:
# ============================================================
# Cell 4. Group duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import group_assets_by_field

unique_id_groups, assets_without_unique_id = group_assets_by_field(
    inventory,
    "photo_library_asset_unique_id",
)

duplicate_candidate_groups = {
    unique_id: group
    for unique_id, group in unique_id_groups.items()
    if len(group) > 1
}

print("assets:", len(inventory["assets"]))
print("generated unique_id count:", len(unique_id_groups))
print("assets without unique_id:", len(assets_without_unique_id))
print("duplicate candidate group count:", len(duplicate_candidate_groups))
print("duplicate candidate asset count:", sum(len(group) for group in duplicate_candidate_groups.values()))

if assets_without_unique_id:
    print()
    print("First assets without unique_id:")
    for asset in assets_without_unique_id[:10]:
        print(
            asset.get("original_filename"),
            asset.get("uuid"),
            asset.get("asset_scope"),
            asset.get("path"),
        )


assets: 71599
generated unique_id count: 71578
assets without unique_id: 0
duplicate candidate group count: 21
duplicate candidate asset count: 42


In [5]:
# ============================================================
# Cell 5. Analyze duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import analyze_duplicate_candidate_groups

duplicate_analysis = analyze_duplicate_candidate_groups(duplicate_candidate_groups)


Analyzing group 1/21
Analyzing group 10/21
Analyzing group 20/21
Analyzing group 21/21
analysis group count: 21
elapsed seconds: 40.614


In [6]:
# ============================================================
# Cell 6. Summary
# ============================================================

from photos_duplicate_cleanup_helpers import count_records_by_status

status_counts = count_records_by_status(duplicate_analysis)

delete_candidate_count = sum(
    len(record.get("delete_candidates") or [])
    for record in duplicate_analysis
)

print("status counts:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print("delete candidate asset count:", delete_candidate_count)

print()
print("First deletable duplicate groups:")
printed = 0

for record in duplicate_analysis:
    if record.get("status") != "DELETABLE_DUPLICATE":
        continue

    print("-" * 80)
    print("reason:", record.get("reason"))
    print("asset_count:", record.get("asset_count"))
    print("keep_assets:", len(record.get("keep_assets") or []))
    print("delete_candidates:", len(record.get("delete_candidates") or []))

    for asset in (record.get("keep_assets") or []):
        print("  KEEP:", asset["original_filename"], asset["date_added"], asset["path"])

    for asset in (record.get("delete_candidates") or []):
        print("  DELETE:", asset["original_filename"], asset["date_added"], asset["path"])

    printed += 1

    if printed >= 10:
        print("... more groups not printed")
        break


status counts:
  DELETABLE_DUPLICATE: 21
delete candidate asset count: 21

First deletable duplicate groups:
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: tmp_v4738122593335909120.mp4 2019-04-14T09:54:23.832120+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/9/96ED2D63-5446-43F7-BED3-4B8E568BCC1A.mp4
  DELETE: tmp_v4738122593335909120.mp4 2019-04-14T09:55:36.001581+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/D/D0DF33BE-8226-4B49-8937-3512A85A2D97.mp4
--------------------------------------------------------------------------------
reason: SAME_UNIQUE_ID_AND_SAME_SHA256
asset_count: 2
keep_assets: 1
delete_candidates: 1
  KEEP: IMG_0209.MOV 2021

In [7]:
# ============================================================
# Cell 7. Write permanent operation report
# ============================================================

import photos_duplicate_cleanup_helpers as cleanup_helpers

REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_result = cleanup_helpers.write_operation_report(
    report_dir=REPORT_DIR,
    duplicate_analysis=duplicate_analysis,
    inventory=inventory,
    assets_without_unique_id=assets_without_unique_id,
    duplicate_candidate_groups=duplicate_candidate_groups,
    run_timestamp=RUN_TIMESTAMP,
    library_id=LIBRARY_ID,
    library_path=LIBRARY_PATH,
    inventory_cache_path=INVENTORY_CACHE_PATH,
)

delete_candidate_rows = report_result["delete_candidate_rows"]
keep_asset_rows = report_result["keep_asset_rows"]
duplicate_review_asset_rows = report_result["duplicate_review_asset_rows"]
location_conflict_rows = report_result["location_conflict_rows"]
live_photo_candidate_rows = report_result["live_photo_candidate_rows"]
assets_without_unique_id_rows = report_result["assets_without_unique_id_rows"]
status_counts = report_result["status_counts"]
safety_counts = report_result["safety_counts"]

print("Wrote report to:", REPORT_DIR)
print("delete_candidate_rows:", len(delete_candidate_rows))
print("keep_asset_rows:", len(keep_asset_rows))
print("duplicate_review_asset_rows:", len(duplicate_review_asset_rows))
print("location_conflict_rows:", len(location_conflict_rows))
print("live_photo_candidate_rows:", len(live_photo_candidate_rows))
print("assets_without_unique_id_rows:", len(assets_without_unique_id_rows))

Photos Library Duplicate Cleanup Report

run_timestamp: 20260608-122355
library_id: backup_20250317
library_path: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
inventory_cache_path: /Users/huohsien/workspace/python/explore_photos_library/data/inventory_cache/backup_20250317.inventory.pkl.gz

asset_count: 71599
assets_without_unique_id: 0
duplicate_candidate_group_count: 21
duplicate_candidate_asset_count: 42

status_counts:
{
  "DELETABLE_DUPLICATE": 21
}

safety_counts:
{
  "live_photo_candidate_group_count": 0,
  "location_conflict_group_count": 0,
  "unreadable_original_group_count": 0,
  "sha_error_group_count": 0
}

delete_candidate_asset_count: 21
keep_asset_count: 21
duplicate_review_asset_count: 42
location_conflict_row_count: 0
live_photo_candidate_row_count: 0
assets_without_unique_id_row_count: 0

Duplicate cleanup v1 decision rule:
- date_added is NOT part of photo_library_asset_uniq

In [8]:
# ============================================================
# Cell 8. Optional: print manual deletion list
# ============================================================
#
# This notebook does NOT delete anything from Photos Library.
# It only produces a report and delete candidate list.
#
# For actual deletion, review delete_candidates.tsv first.

for row in delete_candidate_rows[:100]:
    print(
        row["original_filename"],
        row["date"],
        row["date_added"],
        row["path"],
    )

if len(delete_candidate_rows) > 100:
    print("... more delete candidates not printed")


tmp_v4738122593335909120.mp4 2019-04-14T09:54:04.082973+08:00 2019-04-14T09:55:36.001581+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/D/D0DF33BE-8226-4B49-8937-3512A85A2D97.mp4
IMG_0209.MOV 2021-04-16T18:24:12+08:00 2021-04-16T20:24:05.022936+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/0/03F29B70-3684-48BD-BC61-DA7157512048.mov
IMG_4982.MOV 2022-06-30T11:05:36+08:00 2023-11-05T16:55:08.727065+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary/originals/B/B5F8E445-EFFE-42F0-A46B-D1546C227423.mov
RPReplay_Final1602716023.mp4 2020-10-15T06:53:43+08:00 2020-10-28T14:10:42.324452+08:00 /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by ma

In [9]:
# ============================================================
# Research Cell: inspect IMG_0209.MOV video trim/edit case
# ============================================================

from pathlib import Path
import os
import json
import hashlib
import subprocess
import shutil
import pprint

import osxphotos


LIBRARY_PATH = Path(
    "/Volumes/NEW-PRO-G40--20250315/"
    "Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary"
)

TARGET_UUIDS = [
    "A8B59F76-17D7-4B1C-9AD3-6CEE528944EB",
    "03F29B70-3684-48BD-BC61-DA7157512048",
]

FFPROBE = shutil.which("ffprobe")

print("LIBRARY_PATH:", LIBRARY_PATH)
print("FFPROBE:", FFPROBE)


def safe_attr(obj, name, default=None):
    if not hasattr(obj, name):
        return default

    value = getattr(obj, name)

    if callable(value):
        try:
            return value()
        except TypeError:
            return value
        except Exception as error:
            return f"<ERROR calling {name}: {error}>"

    return value


def to_string(value):
    if value is None:
        return None

    if hasattr(value, "isoformat"):
        return value.isoformat()

    return str(value)


def sha256_file(path):
    if not path:
        return None

    path = Path(path)

    if not path.exists():
        return None

    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


def ffprobe_video(path):
    if not path:
        return None

    path = Path(path)

    if not path.exists():
        return {
            "path": str(path),
            "exists": False,
        }

    if not FFPROBE:
        return {
            "path": str(path),
            "exists": True,
            "error": "ffprobe not found",
        }

    command = [
        FFPROBE,
        "-v",
        "error",
        "-print_format",
        "json",
        "-show_format",
        "-show_streams",
        str(path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        return {
            "path": str(path),
            "exists": True,
            "returncode": result.returncode,
            "stderr": result.stderr,
        }

    data = json.loads(result.stdout)

    video_streams = [
        stream
        for stream in data.get("streams", [])
        if stream.get("codec_type") == "video"
    ]

    first_video = video_streams[0] if video_streams else {}

    return {
        "path": str(path),
        "exists": True,
        "format_duration": data.get("format", {}).get("duration"),
        "format_size": data.get("format", {}).get("size"),
        "format_bit_rate": data.get("format", {}).get("bit_rate"),
        "video_codec": first_video.get("codec_name"),
        "video_width": first_video.get("width"),
        "video_height": first_video.get("height"),
        "video_duration": first_video.get("duration"),
        "avg_frame_rate": first_video.get("avg_frame_rate"),
        "nb_frames": first_video.get("nb_frames"),
    }


def compact_adjustments(value, max_len=2000):
    if value is None:
        return None

    text = repr(value)

    if len(text) <= max_len:
        return text

    return text[:max_len] + " ... <truncated>"


photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))

for uuid in TARGET_UUIDS:
    print("\n" + "=" * 100)
    print("UUID:", uuid)

    photo = photosdb.get_photo(uuid)

    if photo is None:
        print("NOT FOUND")
        continue

    path = safe_attr(photo, "path")
    path_edited = safe_attr(photo, "path_edited")
    path_derivatives = safe_attr(photo, "path_derivatives")

    row = {
        "uuid": safe_attr(photo, "uuid"),
        "filename": safe_attr(photo, "filename"),
        "original_filename": safe_attr(photo, "original_filename"),

        "isphoto": safe_attr(photo, "isphoto"),
        "ismovie": safe_attr(photo, "ismovie"),

        "date": to_string(safe_attr(photo, "date")),
        "date_added": to_string(safe_attr(photo, "date_added")),
        "date_modified": to_string(safe_attr(photo, "date_modified")),

        "path": str(path) if path else None,
        "path_exists": Path(path).exists() if path else None,
        "path_file_size": os.path.getsize(path) if path and Path(path).exists() else None,
        "path_sha256": sha256_file(path),

        "path_edited": str(path_edited) if path_edited else None,
        "path_edited_exists": Path(path_edited).exists() if path_edited else None,
        "path_edited_file_size": os.path.getsize(path_edited) if path_edited and Path(path_edited).exists() else None,
        "path_edited_sha256": sha256_file(path_edited),

        "path_derivatives": [str(p) for p in path_derivatives] if path_derivatives else [],

        "width": safe_attr(photo, "width"),
        "height": safe_attr(photo, "height"),
        "original_width": safe_attr(photo, "original_width"),
        "original_height": safe_attr(photo, "original_height"),

        "uti": safe_attr(photo, "uti"),
        "uti_original": safe_attr(photo, "uti_original"),
        "uti_edited": safe_attr(photo, "uti_edited"),

        "hasadjustments": safe_attr(photo, "hasadjustments"),
        "adjustment_type": safe_attr(photo, "adjustment_type"),
        "external_edit": safe_attr(photo, "external_edit"),

        "favorite": safe_attr(photo, "favorite"),
        "hidden": safe_attr(photo, "hidden"),
        "description": safe_attr(photo, "description"),
        "keywords": list(safe_attr(photo, "keywords") or []),
        "albums": list(safe_attr(photo, "albums") or []),

        "location": safe_attr(photo, "location"),
        "latitude": safe_attr(photo, "latitude"),
        "longitude": safe_attr(photo, "longitude"),
        "place": safe_attr(photo, "place"),
    }

    pprint.pp(row, width=140)

    print("\n--- ffprobe original path ---")
    pprint.pp(ffprobe_video(path), width=140)

    print("\n--- ffprobe edited path ---")
    pprint.pp(ffprobe_video(path_edited), width=140)

    print("\n--- adjustments repr ---")
    print(compact_adjustments(safe_attr(photo, "adjustments")))

LIBRARY_PATH: /Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on 20260604.photoslibrary
FFPROBE: /usr/local/bin/ffprobe

UUID: A8B59F76-17D7-4B1C-9AD3-6CEE528944EB
{'uuid': 'A8B59F76-17D7-4B1C-9AD3-6CEE528944EB',
 'filename': 'A8B59F76-17D7-4B1C-9AD3-6CEE528944EB.mov',
 'original_filename': 'IMG_0209.MOV',
 'isphoto': False,
 'ismovie': True,
 'date': '2021-04-16T18:24:12+08:00',
 'date_added': '2021-04-16T18:25:32.792531+08:00',
 'date_modified': None,
 'path': '/Volumes/NEW-PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）--opened by macOS Sequoia on '
         '20260604.photoslibrary/originals/A/A8B59F76-17D7-4B1C-9AD3-6CEE528944EB.mov',
 'path_exists': True,
 'path_file_size': 884844309,
 'path_sha256': 'a0b2cc278d4b1c8673d2c496192f11fd83489bd558a929d0fb4062180ffdf5b5',
 'path_edited': None,
 'path_edited_exists': None,
 'path_edited_file_size': None,
 'path_edited_sha256': None,
 'path_derivati